In [22]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [3]:
def load_tokenizer(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Tokenizer file not found: {path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Restore ordered BPE merge rules.
    merges = {
        (first_id, second_id): new_id
        for first_id, second_id, new_id in data["merges"]
    }

    vocab = build_vocab(merges)

    return merges, vocab, data["vocab_size"]

In [4]:
tokenizer_path = Path("tokenizer/tokenizer/tokenizer.json")
if not tokenizer_path.exists():
    tokenizer_path = Path("..") / tokenizer_path

merges, vocab, vocab_size = load_tokenizer(tokenizer_path)

print("Tokenizer loaded")
print("Vocabulary size:", vocab_size)
print("Number of merges:", len(merges))

Tokenizer loaded
Vocabulary size: 1000
Number of merges: 744


In [5]:
sample = "This is a tokenizer test."

token_ids = encode(sample, merges)
reconstructed = decode(token_ids, vocab)

print("Token IDs:", token_ids)
print("Decoded:", reconstructed)
print("Exact match:", sample == reconstructed)

Token IDs: [84, 398, 445, 289, 313, 107, 276, 105, 122, 292, 938, 304, 46]
Decoded: This is a tokenizer test.
Exact match: True


In [6]:
data_path = Path("tinystories_100mb.jsonl")
if not data_path.exists():
    data_path = Path("..") / data_path

all_data = pd.read_json(data_path, lines=True)

text = all_data["text"].dropna().astype(str).str.cat(sep="\n") + "\n"

In [10]:
len(text)

101855429

In [7]:
### encoding entire data 
token_ids = encode(text[:990000], merges)

In [16]:
context_length = 100

temp = token_ids[:1000]

X = []
y = []

for i in range(len(temp) - context_length):
    x = temp[i : i + context_length]
    target = temp[i + 1 : i + context_length + 1]

    X.append(x)
    y.append(target)

In [21]:
print(f"Decoded X[1] \n: {decode(X[1], vocab)}")
print(f"Decoded y[1] \n: {decode(y[1], vocab)}")

Decoded X[1] 
: y, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we c
Decoded y[1] 
: , a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can


- All Set for transformer 

In [ ]:
## Building Token Embedding Matrix
def token_embedding_matrix(vocab_size, embedding_dim):
    # Initialize the embedding matrix with random values
    embedding_matrix = np.random.rand(vocab_size, embedding_dim)

    # Normalize the embeddings to have unit length
    norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
    embedding_matrix /= norms

    return embedding_matrix